This is to explore crypto dataset

# Cryptocurrency Price Prediction: Model Comparison and Interactive Visualization

This notebook demonstrates how to predict cryptocurrency prices using various regression models. The analysis includes:

1. Data loading and feature engineering
2. Training and comparing multiple models (LightGBM, XGBoost, Random Forest, Gradient Boosting, ElasticNet)
3. Analyzing model performance metrics and feature importance
4. Creating interactive visualizations for cryptocurrency price predictions with configurable symbols
5. Building an advanced dashboard with trading insights and error analysis

The goal is to identify the most effective model for cryptocurrency price prediction and visualize the results in an interactive manner.

In [1]:
!pip install lightgbm pandas numpy scikit-learn matplotlib optuna xgboost seaborn ipywidgets tensorflow keras

You should consider upgrading via the '/Users/soumyadey/.pyenv/versions/3.10.0/bin/python3.10 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/soumyadey/.pyenv/versions/3.10.0/bin/python3.10 -m pip install --upgrade pip' command.


In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Load and preview the data
df = pd.read_csv('../data/crypto_cleaned.csv', parse_dates=['date'])
df = df.sort_values('date')
df.head()

,slug,symbol,name,date,ranknow,open,high,low,close,volume,market,close_ratio,spread,volume_missing_flag
0,bitcoin,BTC,Bitcoin,2013-04-28,1,135.300000,135.980000,132.100000,134.210000,0.0,1.488567e+09,0.5438,3.88,True
307068,novacoin,NVC,Novacoin,2013-04-28,700,4.220000,4.250000,4.040000,4.250000,0.0,1.162268e+06,1.0000,0.21,True
185144,namecoin,NMC,Namecoin,2013-04-28,371,1.100000,1.120000,1.080000,1.110000,0.0,5.995983e+06,0.7500,0.04,True
386338,terracoin,TRC,Terracoin,2013-04-28,895,0.650793,0.654064,0.634409,0.646892,0.0,1.503099e+06,0.6351,0.02,True
7776,litecoin,LTC,Litecoin,2013-04-28,7,4.300000,4.400000,4.180000,4.350000,0.0,7.463694e+07,0.7727,0.22,True


In [4]:
# Feature engineering
# Spread, return, moving averages, volatility, lag features, and target

# Daily Spread: High - Low
# Daily Return: (Close - Open) / Open
# Moving Averages: 3-day, 7-day, 14-day moving averages of Close
# Volatility: Rolling standard deviation of Close (e.g., 7-day window)
# Relative Strength Index (RSI): 14-day RSI
# Lag Features: Previous day’s Close, Volume, etc.

df['spread'] = df['high'] - df['low']
df['return'] = (df['close'] - df['open']) / df['open']
df['ma_3'] = df['close'].rolling(window=3).mean()
df['ma_7'] = df['close'].rolling(window=7).mean()
df['ma_14'] = df['close'].rolling(window=14).mean()
df['volatility_7'] = df['close'].rolling(window=7).std()
df['close_lag1'] = df['close'].shift(1)
df['volume_lag1'] = df['volume'].shift(1)

# Target: next day's close price
df['target'] = df['close'].shift(-1)

df = df.dropna()
df.head()

,slug,symbol,name,date,ranknow,open,high,low,close,volume,...,spread,volume_missing_flag,return,ma_3,ma_7,ma_14,volatility_7,close_lag1,volume_lag1,target
307070,novacoin,NVC,Novacoin,2013-04-30,700,4.560000,4.69000,4.20000,4.3100,0.0,...,0.4900,True,-0.054825,3.121383,2.354672,21.855437,1.985458,0.67415,0.0,139.000000
2,bitcoin,BTC,Bitcoin,2013-04-30,1,144.000000,146.93000,134.05000,139.0000,0.0,...,12.8800,True,-0.034722,47.994717,22.110430,22.197580,51.576581,4.31000,0.0,1.500000
185146,namecoin,NMC,Namecoin,2013-04-30,371,1.320000,1.72000,1.27000,1.5000,0.0,...,0.4500,True,0.136364,48.270000,22.134716,22.001151,51.565204,139.00000,0.0,0.375300
89418,peercoin,PPC,Peercoin,2013-04-30,159,0.407066,0.43654,0.36314,0.3753,0.0,...,0.0734,True,-0.078036,46.958433,21.521187,21.948673,51.832491,1.50000,0.0,4.300000
7778,litecoin,LTC,Litecoin,2013-04-30,7,4.400000,4.57000,4.17000,4.3000,0.0,...,0.4000,True,-0.022727,2.058433,22.077064,22.209609,51.588627,0.37530,0.0,0.322892


In [5]:
# Split data into train and test sets (time-based split)
features = ['open', 'high', 'low', 'close', 'volume', 'spread', 'return', 'ma_3', 'ma_7', 'ma_14', 'volatility_7', 'close_lag1', 'volume_lag1']
X = df[features]
y = df['target']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 659668, Test size: 164918


In [6]:
# Train LightGBM quantile regression models
params_median = {'objective': 'quantile', 'alpha': 0.5, 'n_estimators': 100, 'random_state': 42}
params_lower = {'objective': 'quantile', 'alpha': 0.1, 'n_estimators': 100, 'random_state': 42}
params_upper = {'objective': 'quantile', 'alpha': 0.9, 'n_estimators': 100, 'random_state': 42}

model_median = lgb.LGBMRegressor(**params_median)
model_lower = lgb.LGBMRegressor(**params_lower)
model_upper = lgb.LGBMRegressor(**params_upper)

model_median.fit(X_train, y_train)
model_lower.fit(X_train, y_train)
model_upper.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 659668, number of used features: 13
[LightGBM] [Info] Start training from score 0.027380
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004892 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 659668, number of used features: 13
[LightGBM] [Info] Start training from score 0.000155
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004892 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'quantile'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [10]:
# Comprehensive Model Training and Comparison
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, explained_variance_score, median_absolute_error
from sklearn.metrics import f1_score, precision_score, recall_score  # For classification metrics if needed
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Import TensorFlow and Keras for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, Concatenate
from tensorflow.keras.layers import Layer, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
import pickle
import os

print("Training and comparing multiple models to find the best one...\n")

# Create a custom attention layer for LSTM
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros")
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        # Alignment scores
        e = K.tanh(K.dot(x, self.W) + self.b)
        # Remove dimension of size 1
        e = K.squeeze(e, axis=-1)
        # Compute the weights
        alpha = K.softmax(e)
        # Reshape to match the input shape
        alpha = K.expand_dims(alpha, axis=-1)
        # Compute the context vector
        context = x * alpha
        context = K.sum(context, axis=1)
        return context

# Create a custom wrapper for LSTM to match scikit-learn API
class EnhancedLSTMRegressor:
    def __init__(self, lstm_units=64, dropout_rate=0.25, learning_rate=0.001, epochs=80, 
                 batch_size=32, verbose=1, sequence_length=10, use_attention=True,
                 bidirectional=True, use_residual=True):
        self.lstm_units = lstm_units
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.sequence_length = sequence_length
        self.use_attention = use_attention
        self.bidirectional = bidirectional
        self.use_residual = use_residual
        self.model = None
        self.scaler_X = MinMaxScaler()  # Changed to MinMaxScaler for better LSTM performance
        self.scaler_y = StandardScaler()
        self.training_history = None
        
    def _create_sequences(self, X):
        """Create sequences for LSTM input with specified sequence length"""
        n_samples = X.shape[0]
        if n_samples <= self.sequence_length:
            # If we don't have enough samples, use what we have
            self.sequence_length = max(1, n_samples - 1)
            print(f"Warning: Reduced sequence length to {self.sequence_length} due to limited samples")
            
        # Number of sequences we can create
        n_sequences = n_samples - self.sequence_length
        
        # Create empty arrays for sequences
        X_seq = np.zeros((n_sequences, self.sequence_length, X.shape[1]))
        
        # Fill the sequences
        for i in range(n_sequences):
            X_seq[i] = X[i:i+self.sequence_length]
            
        return X_seq, n_sequences
        
    def _lr_scheduler(self, epoch):
        """Custom learning rate scheduler with warm-up and decay"""
        # Initial warm-up for 5 epochs
        if epoch < 5:
            return self.learning_rate * (epoch + 1) / 5
        # Then exponential decay
        return self.learning_rate * np.exp(-0.1 * (epoch - 5))
    
    def _create_model(self, input_shape):
        """Create the LSTM model architecture based on configuration (optimized for speed)"""
        inputs = Input(shape=input_shape)
        
        if self.bidirectional:
            # Bidirectional LSTM layers (only if bidirectional=True)
            x = Bidirectional(LSTM(self.lstm_units, return_sequences=False))(inputs)
            x = Dropout(self.dropout_rate)(x)
        else:
            # Use standard LSTM with GPU acceleration if available
            # Modern TF versions use CuDNN automatically when on GPU
            x = LSTM(self.lstm_units, 
                     activation='tanh',
                     recurrent_activation='sigmoid')(inputs)
            x = Dropout(self.dropout_rate)(x)
        
        # Add attention only if specifically requested
        if self.use_attention and hasattr(x, 'shape') and len(x.shape) > 2:
            # If x has sequence dimension (shape length > 2)
            try:
                if hasattr(tf.keras.layers, 'MultiHeadAttention'):
                    attention_output = MultiHeadAttention(
                        num_heads=2,  # Reduced from 4 to 2
                        key_dim=max(16, self.lstm_units//4)  # Ensure key_dim is at least 16
                    )(x, x, x)
                    x = attention_output
                    context_vector = GlobalAveragePooling1D()(x)
                else:
                    context_vector = AttentionLayer()(x)
            except Exception as e:
                print(f"Attention error, falling back to simpler model: {e}")
                context_vector = GlobalAveragePooling1D()(x) if len(x.shape) > 2 else x
        else:
            # No attention - simpler and faster
            context_vector = x
            
        # Simplified dense layers - single layer instead of two
        dense = Dense(32, activation='relu')(context_vector)
        
        # Output layer
        outputs = Dense(1)(dense)
        
        # Create and compile model
        model = Model(inputs=inputs, outputs=outputs)
        
        # Create optimizer with clipnorm to prevent exploding gradients
        optimizer = Adam(
            learning_rate=self.learning_rate,
            clipnorm=1.0
        )
        
        model.compile(optimizer=optimizer, loss='mse')
        return model
        
    def fit(self, X, y):
        # Scale the data
        X_scaled = self.scaler_X.fit_transform(X)
        y_scaled = self.scaler_y.fit_transform(y.values.reshape(-1, 1))
        
        # Create sequences for LSTM
        X_seq, n_sequences = self._create_sequences(X_scaled)
        # The target y values need to be aligned with the sequences
        # Each sequence predicts the y value that follows it
        y_seq = y_scaled[self.sequence_length:]
        
        # Create and compile model
        self.model = self._create_model((self.sequence_length, X.shape[1]))
        
        # Optimized callbacks for faster training
        callbacks = [
            # Early stopping with reduced patience
            EarlyStopping(
                monitor='val_loss',
                patience=8,  # Reduced from 15
                restore_best_weights=True,
                verbose=1
            ),
            # Adaptive learning rate reduction (faster than scheduler)
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.3,  # More aggressive reduction
                patience=3,   # Reduced patience
                min_lr=1e-5,  # Slightly higher minimum to converge faster
                verbose=1
            )
            # Removed LearningRateScheduler for simplicity and speed
        ]
        
        # Apply more compatible performance optimizations
        # Simpler, more compatible approach to optimization
        print("Training model with optimized settings")
        # No GPU-specific optimizations - they'll be handled by TensorFlow automatically
        # This avoids compatibility issues with different TF versions
        
        # Train the model with smaller validation split and early stopping
        self.training_history = self.model.fit(
            X_seq, y_seq,
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=0.1,  # Reduced validation split for faster training
            callbacks=callbacks,
            verbose=self.verbose,
            shuffle=True  # Ensure data is shuffled for better training
            # Removed incompatible parameters: use_multiprocessing and workers
        )
        
        return self
    
    def predict(self, X):
        """Optimized prediction function for faster inference"""
        # Scale input data
        X_scaled = self.scaler_X.transform(X)
        
        # Fast path for sufficient samples
        if X.shape[0] >= self.sequence_length:
            # Create sequences efficiently
            n_sequences = X.shape[0] - self.sequence_length + 1
            X_seq = np.zeros((n_sequences, self.sequence_length, X.shape[1]))
            
            # Efficient sequence creation using a single loop (faster than nested loops)
            for i in range(n_sequences):
                X_seq[i] = X_scaled[i:i + self.sequence_length]
                
            # Predict in batches for speed
            batch_size = min(1024, X_seq.shape[0])  # Use larger batches for prediction
            y_pred_scaled = self.model.predict(X_seq, batch_size=batch_size, verbose=0)
            y_pred = self.scaler_y.inverse_transform(y_pred_scaled).flatten()
            
            # Handle padding efficiently
            predictions = np.full(X.shape[0], np.nan)
            predictions[self.sequence_length-1:self.sequence_length+len(y_pred)-1] = y_pred
            
            # Fast fill NaN values using numpy masked operations
            mask = np.isnan(predictions)
            if mask.any():
                # Fast fill with mean of valid predictions
                valid_mean = np.nanmean(predictions)
                predictions[mask] = valid_mean
                
        else:
            # Fast path for insufficient samples
            # Just predict the last point (simplification for speed)
            last_sequence = np.zeros((1, self.sequence_length, X.shape[1]))
            # Pad with zeros if needed
            seq_length = min(self.sequence_length, X.shape[0])
            last_sequence[0, -seq_length:, :] = X_scaled[-seq_length:]
            
            # Single prediction
            y_pred_scaled = self.model.predict(last_sequence, verbose=0)
            y_pred_value = self.scaler_y.inverse_transform(y_pred_scaled)[0, 0]
            
            # Use the single prediction value for all points
            predictions = np.full(X.shape[0], y_pred_value)
                
        return predictions
    
    def plot_training_history(self):
        """Plot the training history to visualize loss trends"""
        if self.training_history is None:
            print("Model has not been trained yet.")
            return
        
        plt.figure(figsize=(12, 4))
        
        # Plot training & validation loss
        plt.subplot(1, 2, 1)
        plt.plot(self.training_history.history['loss'])
        plt.plot(self.training_history.history['val_loss'])
        plt.title('Model Loss During Training')
        plt.ylabel('Loss')
        plt.xlabel('Epoch')
        plt.legend(['Train', 'Validation'], loc='upper right')
        
        # Plot learning rate if available
        if 'lr' in self.training_history.history:
            plt.subplot(1, 2, 2)
            plt.plot(self.training_history.history['lr'])
            plt.title('Learning Rate')
            plt.ylabel('Learning Rate')
            plt.xlabel('Epoch')
            plt.yscale('log')
        
        plt.tight_layout()
        plt.show()

# Function to calculate Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    """
    Calculate mean absolute percentage error
    Handles zero values in y_true by adding a small epsilon
    """
    epsilon = 1e-10  # Small constant to avoid division by zero
    y_true_safe = np.maximum(np.abs(y_true), epsilon)
    return np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100

# Define models to compare
models = {
    'LightGBM': lgb.LGBMRegressor(objective='regression', random_state=42, n_estimators=100),
    'XGBoost': xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators=100),
    # 'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42, n_estimators=100),
    'ElasticNet': ElasticNet(random_state=42, alpha=0.5, l1_ratio=0.5),
    'LSTM': EnhancedLSTMRegressor(
        lstm_units=64,                # Reduced from 128 to 64 for faster training
        dropout_rate=0.2,             # Slightly reduced dropout
        epochs=40,                    # Reduced number of max epochs
        batch_size=64,                # Increased batch size for faster training
        verbose=1,
        sequence_length=5,            # Reduced sequence length for faster training
        use_attention=False,          # Removed attention for faster training
        bidirectional=False,          # Changed to unidirectional for speed
        use_residual=False            # Removed residual connections for simplicity
    )
}

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Dictionary to store results
model_results = {}
trained_models = {}
predictions = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Make predictions
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    predictions[name] = test_preds
    
    # Calculate standard regression metrics
    train_mae = mean_absolute_error(y_train, train_preds)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    train_r2 = r2_score(y_train, train_preds)
    
    test_mae = mean_absolute_error(y_test, test_preds)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    test_r2 = r2_score(y_test, test_preds)
    
    # Calculate additional regression metrics
    test_mape = mean_absolute_percentage_error(y_test, test_preds)
    test_explained_variance = explained_variance_score(y_test, test_preds)
    test_median_ae = median_absolute_error(y_test, test_preds)
    
    # Store results with expanded metrics
    model_results[name] = {
        'Train MAE': train_mae,
        'Train RMSE': train_rmse,
        'Train R²': train_r2,
        'Test MAE': test_mae,
        'Test RMSE': test_rmse,
        'Test R²': test_r2,
        'Test MAPE': test_mape,
        'Test Explained Variance': test_explained_variance,
        'Test Median AE': test_median_ae
    }
    
    # Add directional accuracy (not F1, but useful for financial time series)
    actual_direction = np.sign(y_test.values[1:] - y_test.values[:-1])
    pred_direction = np.sign(test_preds[1:] - test_preds[:-1])
    direction_match = actual_direction == pred_direction
    direction_accuracy = np.mean(direction_match) * 100
    model_results[name]['Directional Accuracy'] = direction_accuracy
    
    # Save each model to its own pickle file
    model_file = f'../models/{name.replace(" ", "_").lower()}_model.pkl'
    
    # Special handling for LSTM
    if name == 'LSTM':
        # Save weights separately
        if hasattr(model, 'model') and model.model is not None:
            # Create lstm directory if needed
            os.makedirs('../models/lstm', exist_ok=True)
            # Save weights
            model.model.save_weights(f'../models/lstm/weights.h5')
            # Save full model architecture
            model.model.save(f'../models/lstm/full_model')
            print(f"  LSTM weights saved to: ../models/lstm/weights.h5")
            print(f"  LSTM full model saved to: ../models/lstm/full_model")
    
    # Save the model
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    print(f"  Model saved to: {model_file}")
    
    # Save model metadata
    model_metadata = {
        'name': name,
        'metrics': model_results[name],
        'features': features,
        'date_trained': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'is_lstm': name == 'LSTM',
        'lstm_params': {
            'sequence_length': model.sequence_length,
            'lstm_units': model.lstm_units,
            'dropout_rate': model.dropout_rate,
            'use_attention': model.use_attention if hasattr(model, 'use_attention') else False,
            'bidirectional': model.bidirectional if hasattr(model, 'bidirectional') else False,
        } if name == 'LSTM' else {}
    }
    
    with open(f'../models/{name.replace(" ", "_").lower()}_metadata.pkl', 'wb') as f:
        pickle.dump(model_metadata, f)
    
    print(f"  Train - MAE: ${train_mae:.4f}, RMSE: ${train_rmse:.4f}, R²: {train_r2:.4f}")
    print(f"  Test  - MAE: ${test_mae:.4f}, RMSE: ${test_rmse:.4f}, R²: {test_r2:.4f}")
    print(f"  Additional - MAPE: {test_mape:.2f}%, Directional Accuracy: {direction_accuracy:.2f}%")
    print()
    
    # Plot LSTM training history if applicable
    if name == 'LSTM' and hasattr(model, 'plot_training_history'):
        model.plot_training_history()

# Convert results to DataFrame for easier visualization
results_df = pd.DataFrame(model_results).T
results_df = results_df.sort_values('Test MAE')

# Find the best model based on Test MAE
best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]
best_predictions = predictions[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"Test MAE: ${results_df.loc[best_model_name, 'Test MAE']:.4f}")
print(f"Test RMSE: ${results_df.loc[best_model_name, 'Test RMSE']:.4f}")
print(f"Test R²: {results_df.loc[best_model_name, 'Test R²']:.4f}")
print(f"Test MAPE: {results_df.loc[best_model_name, 'Test MAPE']:.2f}%")
print(f"Directional Accuracy: {results_df.loc[best_model_name, 'Directional Accuracy']:.2f}%")

# Save the best model name for reference
with open('../models/best_model_name.txt', 'w') as f:
    f.write(best_model_name)

# Create visualization of model comparison with additional metrics
plt.figure(figsize=(16, 12))

# Plot 1: Test MAE comparison
plt.subplot(2, 3, 1)
sns.barplot(x=results_df.index, y='Test MAE', data=results_df, palette='viridis')
plt.title('Test MAE by Model (Lower is Better)', fontsize=12)
plt.ylabel('Mean Absolute Error ($)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 2: Test RMSE comparison
plt.subplot(2, 3, 2)
sns.barplot(x=results_df.index, y='Test RMSE', data=results_df, palette='viridis')
plt.title('Test RMSE by Model (Lower is Better)', fontsize=12)
plt.ylabel('Root Mean Squared Error ($)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 3: Test R² comparison
plt.subplot(2, 3, 3)
sns.barplot(x=results_df.index, y='Test R²', data=results_df, palette='viridis')
plt.title('Test R² by Model (Higher is Better)', fontsize=12)
plt.ylabel('R² Score')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 4: Test MAPE comparison
plt.subplot(2, 3, 4)
sns.barplot(x=results_df.index, y='Test MAPE', data=results_df, palette='viridis')
plt.title('Test MAPE by Model (Lower is Better)', fontsize=12)
plt.ylabel('Mean Absolute Percentage Error (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 5: Directional Accuracy
plt.subplot(2, 3, 5)
sns.barplot(x=results_df.index, y='Directional Accuracy', data=results_df, palette='viridis')
plt.title('Directional Accuracy by Model (Higher is Better)', fontsize=12)
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 6: Train vs. Test MAE
plt.subplot(2, 3, 6)
train_mae = results_df['Train MAE'].values
test_mae = results_df['Test MAE'].values
model_names = results_df.index

x = np.arange(len(model_names))
width = 0.35

plt.bar(x - width/2, train_mae, width, label='Train MAE')
plt.bar(x + width/2, test_mae, width, label='Test MAE')
plt.xlabel('Model')
plt.ylabel('MAE ($)')
plt.title('Train vs Test MAE by Model')
plt.xticks(x, model_names, rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.suptitle('Enhanced Model Comparison with Additional Metrics', fontsize=16)
plt.subplots_adjust(top=0.9)
plt.show()

# Display feature importance for the best model if available
if best_model_name in ['LightGBM', 'XGBoost', 'Gradient Boosting']:
    plt.figure(figsize=(10, 6))
    
    # Get feature importance
    if hasattr(best_model, 'feature_importances_'):
        feature_importance = best_model.feature_importances_
        sorted_idx = np.argsort(feature_importance)
        plt.barh(np.array(features)[sorted_idx], feature_importance[sorted_idx])
        plt.xlabel('Feature Importance')
        plt.title(f'Feature Importance for {best_model_name}')
        plt.tight_layout()
        plt.show()

# Create a comprehensive comparison table for all metrics
comparison_df = results_df[['Test MAE', 'Test RMSE', 'Test R²', 'Test MAPE', 'Directional Accuracy', 'Test Explained Variance', 'Test Median AE']]
comparison_df = comparison_df.sort_values('Test MAE')  # Sort by MAE

# Style the DataFrame for better visualization
styled_df = comparison_df.style.highlight_min(color='lightgreen', subset=['Test MAE', 'Test RMSE', 'Test MAPE', 'Test Median AE'])
styled_df = styled_df.highlight_max(color='lightgreen', subset=['Test R²', 'Directional Accuracy', 'Test Explained Variance'])
styled_df = styled_df.format({
    'Test MAE': '${:.4f}',
    'Test RMSE': '${:.4f}',
    'Test R²': '{:.4f}',
    'Test MAPE': '{:.2f}%',
    'Directional Accuracy': '{:.2f}%',
    'Test Explained Variance': '{:.4f}',
    'Test Median AE': '${:.4f}'
})

display(styled_df)

Training and comparing multiple models to find the best one...

Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 659668, number of used features: 13
[LightGBM] [Info] Start training from score 406.240092
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004298 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 659668, number of used features: 13
[LightGBM] [Info] Start training from score 406.240092
  Model saved to: ../models/lightgbm_model.pkl
  Train - MAE: $804.8613, RMSE: $14352.3455, R²: 0

/Users/soumyadey/.pyenv/versions/3.10.0/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.390e+13, tolerance: 1.479e+10
  model = cd_fast.enet_coordinate_descent(


  Model saved to: ../models/elasticnet_model.pkl
  Train - MAE: $797.1739, RMSE: $14969.8680, R²: 0.0002
  Test  - MAE: $538.8368, RMSE: $4295.6859, R²: -0.0047
  Additional - MAPE: 704077147.46%, Directional Accuracy: 47.11%

Training LSTM...
Training model with optimized settings
Epoch 1/40
Training model with optimized settings
Epoch 1/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step - loss: 1.0347 - val_loss: 0.2451 - learning_rate: 0.0010
Epoch 2/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step - loss: 1.0347 - val_loss: 0.2451 - learning_rate: 0.0010
Epoch 2/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - loss: 1.1400 - val_loss: 0.2449 - learning_rate: 0.0010
Epoch 3/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - loss: 1.1400 - val_loss: 0.2449 - learning_rate: 0.0010
Epoch 3/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 1.0087 - val_loss: 0.2449 - learning_rate: 0.0010
Epoch 4/40
9277/9277 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 1.0087 - val_loss: 0.2449 - learning_ra

ValueError: The filename must end in `.weights.h5`. Received: filepath=../models/lstm/weights.h5

## Enhanced LSTM for Cryptocurrency Price Prediction

Long Short-Term Memory (LSTM) networks are particularly well-suited for cryptocurrency price prediction, and we've implemented an advanced version with several enhancements to better capture the complex patterns in cryptocurrency markets.

### Why LSTM is Ideal for Cryptocurrency Price Prediction:

1. **Sequential Memory**: LSTMs remember patterns over long sequences, capturing time-dependent price trends and cycles in volatile crypto markets.

2. **Handling Volatility**: Cryptocurrency markets experience rapid price swings. LSTMs can learn both short-term fluctuations and longer-term trends through their sophisticated memory cells.

3. **Non-Linear Pattern Recognition**: Traditional models struggle with the non-linear nature of crypto markets. LSTMs excel at learning complex relationships between features and future prices.

4. **Adaptability**: LSTMs can adapt to changing market conditions by giving more weight to recent patterns while still considering historical context.

### Our Enhanced LSTM Implementation:

Our improved LSTM architecture includes several advanced features specifically designed for financial time series prediction:

#### 1. Attention Mechanism
- **Self-Attention Layer**: Helps the model focus on the most relevant parts of the input sequence
- **Multi-Head Attention**: Allows the model to focus on different representation subspaces at different positions
- **Residual Connections**: Preserves important information through deep networks

#### 2. Bidirectional Architecture
- Processes sequences in both forward and backward directions
- Captures patterns that might be more apparent when viewed from either direction
- Creates richer representations of temporal relationships

#### 3. Advanced Sequence Processing
- **Variable-Length Sequences**: Configurable sequence length parameter (default: 10 days)
- **Dynamic Sequence Creation**: Properly aligns input sequences with target values

#### 4. Optimized Training Process
- **Learning Rate Scheduling**: Custom scheduler with warm-up and exponential decay
- **Early Stopping**: Prevents overfitting by monitoring validation loss
- **Gradient Clipping**: Prevents exploding gradients during training

#### 5. Advanced Model Architecture
- **Multiple LSTM Layers**: Hierarchical feature extraction
- **Dropout Regularization**: Prevents overfitting with strategic neuron deactivation
- **Layer Normalization**: Stabilizes training by normalizing activations
- **MinMax Scaling**: Improved input normalization especially suited for LSTM

The attention mechanism is particularly important as it allows the model to focus more on relevant time steps when making predictions, rather than treating all historical data points equally. This is especially valuable in cryptocurrency markets where certain events or price movements may have outsized importance for future predictions.

For storing and loading this complex model, we've implemented advanced serialization that preserves both the architecture and weights, ensuring consistent predictions across sessions.

In [ ]:
# Save the best model to a pickle file for later reuse
import pickle
import os
import importlib
import sys

# Create a models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the best model to a pickle file
best_model_file = f'../models/{best_model_name}_best_model.pkl'

# Handle special case for LSTM (not directly pickle-able)
lstm_params = {}
if best_model_name == 'LSTM':
    # For LSTM, save the model weights separately using TF's save_weights
    if hasattr(best_model, 'model') and best_model.model is not None:
        # Save both weights file and full model architecture
        best_model.model.save_weights(f'../models/lstm_weights.h5')
        # Save the full model in SavedModel format for better restoration
        best_model.model.save(f'../models/lstm_full_model')
        print(f"LSTM weights saved to: ../models/lstm_weights.h5")
        print(f"LSTM full model saved to: ../models/lstm_full_model")
        
        # Store LSTM-specific parameters
        lstm_params = {
            'sequence_length': best_model.sequence_length,
            'lstm_units': best_model.lstm_units,
            'dropout_rate': best_model.dropout_rate,
            'learning_rate': best_model.learning_rate,
            'batch_size': best_model.batch_size,
            'use_attention': best_model.use_attention if hasattr(best_model, 'use_attention') else False,
            'bidirectional': best_model.bidirectional if hasattr(best_model, 'bidirectional') else False,
            'use_residual': best_model.use_residual if hasattr(best_model, 'use_residual') else False
        }
    else:
        print("Warning: LSTM model doesn't have weights to save")

with open(best_model_file, 'wb') as f:
    pickle.dump(best_model, f)

# Create a comprehensive dependencies dictionary for functions needed in other cells
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython import display  # Import as a module, not a function

dependencies = {
    # Store the required functions directly
    'mean_absolute_error': mean_absolute_error,
    'mean_squared_error': mean_squared_error,
    'r2_score': r2_score,
    # Store imports info for other libraries
    'imports': {
        'pandas': 'pd',
        'numpy': 'np',
        'matplotlib.pyplot': 'plt',
        'seaborn': 'sns',
        'matplotlib.dates.DateFormatter': 'DateFormatter',
        'matplotlib.dates': 'mdates',
        'ipywidgets': 'widgets',
        'IPython.display': 'display',  # This will be imported as a module
        'pickle': 'pickle',
        'os': 'os',
        'tensorflow': 'tf',
        'sklearn.preprocessing.StandardScaler': 'StandardScaler',
        'sklearn.preprocessing.MinMaxScaler': 'MinMaxScaler',
        'tensorflow.keras.layers.MultiHeadAttention': 'MultiHeadAttention',
        'tensorflow.keras.layers.LayerNormalization': 'LayerNormalization'
    }
}

# Save model metadata to a separate file for easier reading
model_metadata = {
    'model_name': best_model_name,
    'features': features,
    'metrics': {
        'test_mae': float(results_df.loc[best_model_name, 'Test MAE']),
        'test_rmse': float(results_df.loc[best_model_name, 'Test RMSE']),
        'test_r2': float(results_df.loc[best_model_name, 'Test R²']),
    },
    'date_trained': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    # Include comprehensive dependencies for other cells
    'dependencies': dependencies,
    # Include LSTM specific information if that's the best model
    'is_lstm': best_model_name == 'LSTM',
    'lstm_params': lstm_params,
    'is_enhanced_lstm': hasattr(best_model, 'use_attention') if best_model_name == 'LSTM' else False
}

with open('../models/model_metadata.pkl', 'wb') as f:
    pickle.dump(model_metadata, f)

print(f"Best model ({best_model_name}) saved to: {best_model_file}")
print(f"Model metadata and dependencies saved to: ../models/model_metadata.pkl")
print(f"Model features: {features}")
print(f"Required dependencies have been saved with the model metadata.")

if best_model_name == 'LSTM':
    print(f"\nLSTM Configuration:")
    print(f"Sequence Length: {best_model.sequence_length}")
    print(f"LSTM Units: {best_model.lstm_units}")
    print(f"Dropout Rate: {best_model.dropout_rate}")
    print(f"Batch Size: {best_model.batch_size}")
    
    # Print enhanced LSTM parameters
    if hasattr(best_model, 'use_attention'):
        print(f"Using Attention: {'Yes' if best_model.use_attention else 'No'}")
    if hasattr(best_model, 'bidirectional'):
        print(f"Bidirectional: {'Yes' if best_model.bidirectional else 'No'}")
    if hasattr(best_model, 'use_residual'):
        print(f"Using Residual Connections: {'Yes' if best_model.use_residual else 'No'}")
        
    # Show training time if available
    if hasattr(best_model, 'training_history'):
        epochs_trained = len(best_model.training_history.history.get('loss', []))
        print(f"Epochs Trained: {epochs_trained}")
        final_val_loss = best_model.training_history.history.get('val_loss', [])[-1] if best_model.training_history.history.get('val_loss') else None
        if final_val_loss:
            print(f"Final Validation Loss: {final_val_loss:.6f}")

LSTM weights saved to: ../models/lstm.weights.h5
Best model (LSTM) saved to: ../models/LSTM_best_model.pkl
Model metadata and dependencies saved to: ../models/model_metadata.pkl
Model features: ['open', 'high', 'low', 'close', 'volume', 'spread', 'return', 'ma_3', 'ma_7', 'ma_14', 'volatility_7', 'close_lag1', 'volume_lag1']
Required dependencies have been saved with the model metadata.
Note: IPython.display is now correctly saved as a module, not a function.

LSTM Configuration:
Sequence Length: 5
LSTM Units: 64
Dropout Rate: 0.2
Batch Size: 32


In [ ]:
# Function to load models from pickle files
import pickle
import os
import importlib
import sys
import numpy as np
from sklearn.metrics import explained_variance_score, median_absolute_error

def load_model(model_name=None, force_reload=False):
    """
    Load a specific trained model from its pickle file.
    If model_name is None, loads the best model.
    If the model is already loaded in memory, it will return that unless force_reload=True.
    Also loads and makes available all required dependencies for predictions.
    
    Args:
        model_name: Name of the model to load. If None, loads the best model.
        force_reload: If True, always reload the model from disk even if it's in memory
        
    Returns:
        model: The loaded model
        model_name: Name of the model
        model_metadata: Dictionary with model metadata
    """
    global best_model, best_model_name
    # Create references to the metrics functions in the global namespace
    global mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
    
    # If model_name is None, load the best model
    if model_name is None:
        # Try to read the best model name from file
        try:
            with open('../models/best_model_name.txt', 'r') as f:
                model_name = f.read().strip()
                print(f"Loading best model: {model_name}")
        except FileNotFoundError:
            print("No best model name file found. Please train models first.")
            return None, None, None
    
    # Convert model name to filename format
    model_filename = model_name.replace(" ", "_").lower()
    
    # If the model is already loaded and we don't need to force reload, return it
    if 'best_model' in globals() and best_model_name == model_name and not force_reload:
        print(f"Using already loaded {best_model_name} model from memory")
        
        # Load metadata
        try:
            with open(f'../models/{model_filename}_metadata.pkl', 'rb') as f:
                model_metadata = pickle.load(f)
        except Exception as e:
            print(f"Error loading metadata: {e}")
            model_metadata = None
            
        return best_model, best_model_name, model_metadata
    
    # Otherwise, load from disk
    try:
        # Load metadata
        with open(f'../models/{model_filename}_metadata.pkl', 'rb') as f:
            model_metadata = pickle.load(f)
            
        model_file = f'../models/{model_filename}_model.pkl'
        
        # Load the actual model
        with open(model_file, 'rb') as f:
            model = pickle.load(f)
        
        # Special handling for LSTM models
        if model_metadata.get('is_lstm', False):
            # Import TensorFlow if needed
            if 'tensorflow' not in sys.modules:
                import tensorflow as tf
                globals()['tf'] = tf
                
            # Make sure required imports are available for enhanced LSTM
            if model_metadata.get('lstm_params', {}).get('use_attention', False):
                try:
                    from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization
                    globals()['MultiHeadAttention'] = MultiHeadAttention
                    globals()['LayerNormalization'] = LayerNormalization
                except ImportError:
                    print("Warning: Enhanced LSTM features may not be fully available with current TensorFlow version")
            
            # Ensure the model has been initialized
            try:
                # Apply saved LSTM parameters from metadata to ensure consistency
                if 'lstm_params' in model_metadata:
                    params = model_metadata['lstm_params']
                    # Set parameters that need updating
                    if hasattr(model, 'sequence_length') and 'sequence_length' in params:
                        print(f"Using sequence length from metadata: {params['sequence_length']}")
                        model.sequence_length = params['sequence_length']
                
                # First ensure the scalers are initialized
                dummy_X = np.zeros((1, len(model_metadata['features'])))
                dummy_y = np.zeros((1,))
                
                if hasattr(model, 'scaler_X'):
                    model.scaler_X.fit(dummy_X)
                if hasattr(model, 'scaler_y'):
                    model.scaler_y.fit(dummy_y.reshape(-1, 1))
                
                # Try to load from full model save first
                if os.path.exists(f'../models/lstm/full_model'):
                    try:
                        from tensorflow.keras.models import load_model
                        model.model = load_model(f'../models/lstm/full_model')
                        print("Successfully loaded full LSTM model architecture and weights")
                    except Exception as e:
                        print(f"Error loading full model: {e}")
                        print("Falling back to weights-only loading")
                
                # If full model loading failed, try weights-only approach
                if not hasattr(model, 'model') or model.model is None:
                    # Rebuild model architecture by forcing a prediction first
                    dummy_pred = model.predict(dummy_X)
                    
                    # Load weights if available
                    if os.path.exists(f'../models/lstm/weights.h5'):
                        model.model.load_weights(f'../models/lstm/weights.h5')
                        print(f"LSTM weights loaded successfully")
                
                # Print LSTM configuration
                if hasattr(model, 'sequence_length'):
                    print(f"LSTM sequence length: {model.sequence_length}")
                if hasattr(model, 'lstm_units'):
                    print(f"LSTM units: {model.lstm_units}")
                    
                # Print enhanced configuration if available
                if model_metadata['lstm_params'].get('use_attention', False):
                    print(f"Using attention: {'Yes' if model.use_attention else 'No'}")
                if model_metadata['lstm_params'].get('bidirectional', False):
                    print(f"Bidirectional: {'Yes' if model.bidirectional else 'No'}")
                
            except Exception as lstm_err:
                print(f"Error loading LSTM model: {lstm_err}")
        
        print(f"Successfully loaded {model_name} model from {model_file}")
        print(f"Model was trained on: {model_metadata['date_trained']}")
        print(f"Test MAE: ${model_metadata['metrics']['test_mae']:.4f}")
        
        if 'test_mape' in model_metadata['metrics']:
            print(f"Test MAPE: {model_metadata['metrics']['test_mape']:.2f}%")
        
        if 'directional_accuracy' in model_metadata['metrics']:
            print(f"Directional Accuracy: {model_metadata['metrics']['directional_accuracy']:.2f}%")
        
        # Make sure we have required functions
        if 'mean_absolute_error' not in globals():
            from sklearn.metrics import mean_absolute_error
            globals()['mean_absolute_error'] = mean_absolute_error
            
        if 'mean_squared_error' not in globals():
            from sklearn.metrics import mean_squared_error
            globals()['mean_squared_error'] = mean_squared_error
            
        if 'r2_score' not in globals():
            from sklearn.metrics import r2_score
            globals()['r2_score'] = r2_score
        
        if 'explained_variance_score' not in globals():
            from sklearn.metrics import explained_variance_score
            globals()['explained_variance_score'] = explained_variance_score
            
        if 'median_absolute_error' not in globals():
            from sklearn.metrics import median_absolute_error
            globals()['median_absolute_error'] = median_absolute_error
            
        if 'mean_absolute_percentage_error' not in globals():
            # Define the function if it doesn't exist
            def mean_absolute_percentage_error(y_true, y_pred):
                epsilon = 1e-10
                y_true_safe = np.maximum(np.abs(y_true), epsilon)
                return np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100
            globals()['mean_absolute_percentage_error'] = mean_absolute_percentage_error
        
        # Import common modules if needed
        if 'np' not in globals():
            import numpy as np
            globals()['np'] = np
            
        if 'pd' not in globals():
            import pandas as pd
            globals()['pd'] = pd
            
        if 'plt' not in globals():
            import matplotlib.pyplot as plt
            globals()['plt'] = plt
            
        # Update global variables
        best_model = model
        best_model_name = model_name
        
        return model, model_name, model_metadata
    
    except Exception as e:
        print(f"Error loading model {model_name}: {e}")
        print("Please train the model first or check if the model file exists.")
        return None, None, None

def load_best_model(force_reload=False):
    """
    Wrapper function to load the best model for backward compatibility
    """
    return load_model(model_name=None, force_reload=force_reload)

def compare_all_models(test_data_x=None, test_data_y=None, plot_predictions=True):
    """
    Load all trained models and compare their performance with comprehensive metrics
    
    Args:
        test_data_x: Optional test features. If None, uses the global X_test
        test_data_y: Optional test targets. If None, uses the global y_test
        plot_predictions: Whether to plot predictions from all models
        
    Returns:
        DataFrame with model comparison
    """
    # Use global test data if none provided
    if test_data_x is None:
        if 'X_test' not in globals():
            print("No test data available. Please provide test_data_x or ensure X_test exists.")
            return None
        test_data_x = globals()['X_test']
    
    if test_data_y is None:
        if 'y_test' not in globals():
            print("No test data available. Please provide test_data_y or ensure y_test exists.")
            return None
        test_data_y = globals()['y_test']
    
    # Find all model files
    model_files = [f for f in os.listdir('../models') if f.endswith('_model.pkl')]
    if not model_files:
        print("No trained models found in ../models directory")
        return None
    
    # Extract model names
    model_names = [f.replace('_model.pkl', '').replace('_', ' ').title() for f in model_files]
    
    # Create empty results dataframe
    results = {}
    all_predictions = {}
    
    print(f"Found {len(model_names)} models to compare:")
    
    # Load each model and evaluate
    for name in model_names:
        print(f"Loading and evaluating {name}...")
        model, _, metadata = load_model(name, force_reload=True)
        
        if model is not None:
            # Make predictions
            predictions = model.predict(test_data_x)
            all_predictions[name] = predictions
            
            # Calculate metrics
            mae = mean_absolute_error(test_data_y, predictions)
            rmse = np.sqrt(mean_squared_error(test_data_y, predictions))
            r2 = r2_score(test_data_y, predictions)
            mape = mean_absolute_percentage_error(test_data_y, predictions)
            
            # Additional metrics
            exp_var = explained_variance_score(test_data_y, predictions)
            median_ae = median_absolute_error(test_data_y, predictions)
            
            # Calculate directional accuracy
            actual_direction = np.sign(test_data_y.values[1:] - test_data_y.values[:-1])
            pred_direction = np.sign(predictions[1:] - predictions[:-1])
            direction_accuracy = np.mean(actual_direction == pred_direction) * 100
            
            # Store results
            results[name] = {
                'MAE': mae,
                'RMSE': rmse,
                'R²': r2,
                'MAPE (%)': mape,
                'Directional Accuracy (%)': direction_accuracy,
                'Explained Variance': exp_var,
                'Median AE': median_ae
            }
    
    if not results:
        print("No models were successfully evaluated")
        return None
    
    # Convert to DataFrame and sort by MAE
    results_df = pd.DataFrame(results).T
    results_df = results_df.sort_values('MAE')
    
    # Style the DataFrame for better visualization
    styled_df = results_df.style.highlight_min(color='lightgreen', subset=['MAE', 'RMSE', 'MAPE (%)', 'Median AE'])
    styled_df = styled_df.highlight_max(color='lightgreen', subset=['R²', 'Directional Accuracy (%)', 'Explained Variance'])
    styled_df = styled_df.format({
        'MAE': '${:.4f}',
        'RMSE': '${:.4f}',
        'R²': '{:.4f}',
        'MAPE (%)': '{:.2f}%',
        'Directional Accuracy (%)': '{:.2f}%',
        'Explained Variance': '{:.4f}',
        'Median AE': '${:.4f}'
    })
    
    # Plot predictions comparison if requested
    if plot_predictions and all_predictions:
        # Get first and last few dates of the test data for x-axis labels
        if 'df' in globals() and 'date' in globals()['df']:
            split_idx = int(len(globals()['df']) * 0.8)
            dates = globals()['df']['date'].iloc[split_idx:].reset_index(drop=True)
        else:
            # Create a dummy date range if dates are not available
            dates = pd.date_range(start='2023-01-01', periods=len(test_data_y))
        
        plt.figure(figsize=(12, 8))
        
        # Plot actual values
        plt.plot(dates, test_data_y, 'k-', linewidth=2.5, alpha=0.7, label='Actual')
        
        # Plot predictions for each model
        colors = plt.cm.tab10.colors
        for i, (name, pred) in enumerate(all_predictions.items()):
            plt.plot(dates, pred, color=colors[i % len(colors)], alpha=0.8, 
                     linestyle='-', label=f'{name} Predictions')
        
        plt.title('Price Predictions Comparison', fontsize=16)
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Price', fontsize=12)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
        # Plot prediction errors
        plt.figure(figsize=(12, 8))
        for i, (name, pred) in enumerate(all_predictions.items()):
            errors = test_data_y.values - pred
            plt.plot(dates, errors, color=colors[i % len(colors)], alpha=0.8, 
                     linestyle='-', label=f'{name} Error')
            
        plt.title('Prediction Errors by Model', fontsize=16)
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Error (Actual - Predicted)', fontsize=12)
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    return styled_df

# Test the loading function
if os.path.exists('../models'):
    model_files = [f for f in os.listdir('../models') if f.endswith('_model.pkl')]
    if model_files:
        print("Found trained models:")
        for file in model_files:
            print(f"- {file.replace('_model.pkl', '').replace('_', ' ').title()}")
        
        loaded_model, loaded_model_name, metadata = load_best_model()
        if loaded_model is not None:
            print(f"\nLoaded model features: {metadata['features']}")
            
            # Display model metrics
            if 'metrics' in metadata:
                print("\nModel metrics:")
                for metric_name, metric_value in metadata['metrics'].items():
                    # Format the output based on the metric type
                    if 'mae' in metric_name.lower() or 'rmse' in metric_name.lower() or 'median' in metric_name.lower():
                        print(f"- {metric_name}: ${metric_value:.4f}")
                    elif 'mape' in metric_name.lower() or 'directional' in metric_name.lower() or 'accuracy' in metric_name.lower():
                        print(f"- {metric_name}: {metric_value:.2f}%")
                    else:
                        print(f"- {metric_name}: {metric_value:.4f}")
            
            # Special verification for LSTM model
            if metadata.get('is_lstm', False):
                print("\n✓ LSTM model loaded with weights")
                if 'lstm_params' in metadata:
                    print(f"✓ LSTM parameters loaded from metadata")
                    # Display key LSTM parameters
                    if 'sequence_length' in metadata['lstm_params']:
                        print(f"  - Sequence Length: {metadata['lstm_params']['sequence_length']}")
                    if 'use_attention' in metadata['lstm_params']:
                        print(f"  - Using Attention: {'Yes' if metadata['lstm_params']['use_attention'] else 'No'}")
                    if 'bidirectional' in metadata['lstm_params']:
                        print(f"  - Bidirectional: {'Yes' if metadata['lstm_params']['bidirectional'] else 'No'}")
    else:
        print("No saved models found. Run the model training cell first.")
else:
    print("Models directory not found. Run the model training cell first.")

Using already loaded LSTM model from memory

Loaded model features: ['open', 'high', 'low', 'close', 'volume', 'spread', 'return', 'ma_3', 'ma_7', 'ma_14', 'volatility_7', 'close_lag1', 'volume_lag1']

Required dependencies have been loaded to the global namespace.
✓ mean_absolute_error is available
✓ mean_squared_error is available
✓ r2_score is available
✓ LSTM model loaded with weights
✓ LSTM parameters loaded from metadata
  - Sequence Length: 5


## Dependency Injection for Model Persistence

This notebook uses a robust approach to model persistence that includes dependency injection. This means:

1. **Complete Dependency Management**: When the model is saved, all required dependencies (like metric functions and imported modules) are saved with it.
2. **Global Namespace Injection**: When the model is loaded, these dependencies are automatically injected into the global namespace.
3. **Resilient Execution**: Any cell can be run independently after loading the model, even if the kernel was reset.

This approach ensures that:
- All visualization and prediction cells can work without relying on the execution of previous cells
- Functions like `mean_absolute_error` and `mean_squared_error` are always available when needed
- The notebook maintains a consistent state regarding model usage

The `load_best_model()` function handles all of this automatically, making the notebook more robust and easier to use.

In [ ]:
# Run comprehensive comparison of all trained models
print("Comparing all trained models with all metrics...")
comparison_results = compare_all_models(plot_predictions=True)

# Display the styled comparison table
if comparison_results is not None:
    display(comparison_results)
    
    # Create a figure to show all metrics side by side
    metrics = ['MAE', 'RMSE', 'R²', 'MAPE (%)', 'Directional Accuracy (%)', 'Explained Variance', 'Median AE']
    model_names = comparison_results.index
    
    plt.figure(figsize=(18, 12))
    
    # Create subplots for each metric
    for i, metric in enumerate(metrics):
        plt.subplot(3, 3, i+1)
        
        # Get the values for this metric
        values = comparison_results.data[metric].values
        
        # For metrics where lower is better, invert the colors
        if metric in ['MAE', 'RMSE', 'MAPE (%)', 'Median AE']:
            colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(values)))
        else:
            colors = plt.cm.RdYlGn(np.linspace(0, 1, len(values)))
            
        # Sort the data for better visualization
        sorted_idx = np.argsort(values)
        if metric in ['R²', 'Directional Accuracy (%)', 'Explained Variance']:
            # For these metrics, higher is better
            sorted_idx = sorted_idx[::-1]
            
        plt.bar(range(len(model_names)), values[sorted_idx], color=colors[sorted_idx])
        plt.xticks(range(len(model_names)), np.array(model_names)[sorted_idx], rotation=45, ha='right')
        plt.title(f"{metric} by Model")
        plt.grid(axis='y', alpha=0.3)
        
        # Add value labels on top of bars
        for j, v in enumerate(values[sorted_idx]):
            if metric in ['MAE', 'RMSE', 'Median AE']:
                plt.text(j, v + 0.02, f"${v:.2f}", ha='center', va='bottom', fontsize=9)
            elif metric in ['MAPE (%)', 'Directional Accuracy (%)']:
                plt.text(j, v + 0.02, f"{v:.1f}%", ha='center', va='bottom', fontsize=9)
            else:
                plt.text(j, v + 0.02, f"{v:.3f}", ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.suptitle("Comprehensive Model Comparison Across All Metrics", fontsize=16, y=1.02)
    plt.show()
    
    # Print summary of best models per metric
    print("\n🏆 Best Model Summary:")
    for metric in metrics:
        if metric in ['MAE', 'RMSE', 'MAPE (%)', 'Median AE']:
            best_model = comparison_results.data.sort_values(metric).index[0]
            best_value = comparison_results.data[metric].min()
            if metric in ['MAE', 'RMSE', 'Median AE']:
                print(f"- Best {metric}: {best_model} (${best_value:.4f})")
            else:
                print(f"- Best {metric}: {best_model} ({best_value:.2f}%)")
        else:
            best_model = comparison_results.data.sort_values(metric, ascending=False).index[0]
            best_value = comparison_results.data[metric].max()
            if metric == 'Directional Accuracy (%)':
                print(f"- Best {metric}: {best_model} ({best_value:.2f}%)")
            else:
                print(f"- Best {metric}: {best_model} ({best_value:.4f})")
else:
    print("No comparison results available. Please train models first.")

In [31]:
# Interactive visualization for price prediction by symbol
import ipywidgets as widgets
from IPython.display import display as ipython_display  # Fix the display import

# First, load the model and required dependencies
loaded_model, loaded_model_name, metadata = load_best_model()

# Ensure we have the required metric functions - import if not available
if 'mean_absolute_error' not in globals():
    from sklearn.metrics import mean_absolute_error
if 'mean_squared_error' not in globals():
    from sklearn.metrics import mean_squared_error
if 'r2_score' not in globals():
    from sklearn.metrics import r2_score
if 'np' not in globals():
    import numpy as np
if 'plt' not in globals():
    import matplotlib.pyplot as plt
if 'mdates' not in globals():
    import matplotlib.dates as mdates
if 'DateFormatter' not in globals():
    from matplotlib.dates import DateFormatter

# Load original data again to ensure we have all symbols
full_df = pd.read_csv('../crypto-markets.csv', parse_dates=['date'])

# Get unique symbols
symbols = sorted(full_df['symbol'].unique())

# Feature engineering function to prepare data for a specific symbol
def prepare_symbol_data(symbol_df):
    # Make a copy to avoid modifying the original
    df_prep = symbol_df.copy()
    
    # Sort by date
    df_prep = df_prep.sort_values('date')
    
    # Feature engineering (same as before)
    df_prep['spread'] = df_prep['high'] - df_prep['low']
    df_prep['return'] = (df_prep['close'] - df_prep['open']) / df_prep['open']
    df_prep['ma_3'] = df_prep['close'].rolling(window=3).mean()
    df_prep['ma_7'] = df_prep['close'].rolling(window=7).mean()
    df_prep['ma_14'] = df_prep['close'].rolling(window=14).mean()
    df_prep['volatility_7'] = df_prep['close'].rolling(window=7).std()
    df_prep['close_lag1'] = df_prep['close'].shift(1)
    df_prep['volume_lag1'] = df_prep['volume'].shift(1)
    
    # Target: next day's close price
    df_prep['target'] = df_prep['close'].shift(-1)
    
    # Drop rows with NaN values
    df_prep = df_prep.dropna()
    
    return df_prep

# Function to create visualization for a selected symbol
def visualize_symbol(symbol):
    # Filter data for the selected symbol
    symbol_df = full_df[full_df['symbol'] == symbol].copy()
    
    if len(symbol_df) < 30:  # Ensure we have enough data
        print(f"Not enough data for {symbol}. Please select another symbol.")
        return
    
    # Prepare data
    prepared_df = prepare_symbol_data(symbol_df)
    
    # Split data (80% train, 20% test)
    split_idx = int(len(prepared_df) * 0.8)
    
    # Extract features and target
    X_symbol = prepared_df[metadata['features']]  # Use features from metadata
    y_symbol = prepared_df['target']
    
    X_train_symbol = X_symbol.iloc[:split_idx]
    X_test_symbol = X_symbol.iloc[split_idx:]
    y_train_symbol = y_symbol.iloc[:split_idx]
    y_test_symbol = y_symbol.iloc[split_idx:]
    
    # Use the loaded model
    model = loaded_model
    
    # Make predictions
    y_pred = model.predict(X_test_symbol)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_symbol, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_symbol, y_pred))
    r2 = r2_score(y_test_symbol, y_pred)
    
    # Get dates for plotting
    test_dates = prepared_df.iloc[split_idx:].date.values[:len(y_test_symbol)]
    
    # Create plot
    plt.figure(figsize=(14, 8))
    
    # Plot actual and predicted prices
    plt.plot(test_dates, y_test_symbol.values, label='Actual Price', color='blue', linewidth=2)
    plt.plot(test_dates, y_pred, label='Predicted Price', color='red', linewidth=2, linestyle='--')
    
    # Add a shaded area for the prediction error
    plt.fill_between(test_dates, 
                    y_test_symbol.values - mae, 
                    y_test_symbol.values + mae, 
                    alpha=0.2, color='gray',
                    label=f'Error Margin (±${mae:.2f})')
    
    # Set title and labels
    plt.title(f'{symbol} Price Prediction using {loaded_model_name}', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price ($)', fontsize=12)
    
    # Add metrics as text
    metrics_text = f"MAE: ${mae:.4f}\nRMSE: ${rmse:.4f}\nR²: {r2:.4f}"
    plt.annotate(metrics_text, xy=(0.02, 0.95), xycoords='axes fraction', 
                bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.8),
                fontsize=10)
    
    # Format x-axis dates
    plt.gcf().autofmt_xdate()
    
    # Add grid and legend
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print performance summary
    print(f"\nPerformance Summary for {symbol}:")
    print(f"Number of data points: {len(prepared_df)}")
    print(f"Training samples: {len(X_train_symbol)}")
    print(f"Testing samples: {len(X_test_symbol)}")
    print(f"Mean Absolute Error (MAE): ${mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    
    # Price range analysis
    min_price = min(y_test_symbol.min(), y_pred.min())
    max_price = max(y_test_symbol.max(), y_pred.max())
    price_range = max_price - min_price
    
    print(f"\nPrice Range Analysis:")
    print(f"Minimum Price: ${min_price:.2f}")
    print(f"Maximum Price: ${max_price:.2f}")
    print(f"Price Range: ${price_range:.2f}")
    print(f"MAE as percentage of price range: {(mae/price_range)*100:.2f}%")

# Create dropdown widget for symbol selection
symbol_dropdown = widgets.Dropdown(
    options=symbols,
    description='Symbol:',
    style={'description_width': 'initial'}
)

# Create button to trigger visualization
visualize_button = widgets.Button(
    description='Visualize',
    button_style='primary',
    tooltip='Click to visualize selected symbol'
)

# Output area for visualization
output = widgets.Output()

# Function to handle button click
def on_button_click(b):
    with output:
        output.clear_output()
        visualize_symbol(symbol_dropdown.value)

# Attach the function to the button
visualize_button.on_click(on_button_click)

# Display widgets
print("Select a cryptocurrency symbol and click 'Visualize' to see price predictions:")
ipython_display(widgets.VBox([symbol_dropdown, visualize_button, output]))  # Use ipython_display instead of display

Using already loaded LSTM model from memory
Select a cryptocurrency symbol and click 'Visualize' to see price predictions:


In [ ]:
# Validate that all models are saved correctly with their metadata
import os
import glob
import pickle

# Check the models directory exists
if not os.path.exists('../models'):
    os.makedirs('../models', exist_ok=True)
    print("Models directory created at ../models")
else:
    print("✓ Models directory exists")

# Find all model files
model_files = glob.glob('../models/*_model.pkl')
metadata_files = glob.glob('../models/*_metadata.pkl')

print(f"\nFound {len(model_files)} model files:")
for model_file in model_files:
    model_name = os.path.basename(model_file).replace('_model.pkl', '').replace('_', ' ').title()
    metadata_file = model_file.replace('_model.pkl', '_metadata.pkl')
    
    # Check if metadata exists
    if os.path.exists(metadata_file):
        # Load metadata to validate
        try:
            with open(metadata_file, 'rb') as f:
                metadata = pickle.load(f)
                
            # Check metadata contains required fields
            has_features = 'features' in metadata
            has_metrics = 'metrics' in metadata
            
            print(f"✓ {model_name} - Model & Metadata OK")
            print(f"  - Features: {len(metadata['features']) if has_features else 'Missing'} features")
            
            if has_metrics:
                # Print key metrics
                print(f"  - Test MAE: ${metadata['metrics'].get('test_mae', metadata['metrics'].get('Test MAE', 'N/A'))}")
                
                # Check for additional metrics
                metrics = metadata['metrics']
                additional_metrics = []
                
                if 'Test MAPE' in metrics or 'test_mape' in metrics:
                    additional_metrics.append('MAPE')
                if 'Directional Accuracy' in metrics or 'directional_accuracy' in metrics:
                    additional_metrics.append('Directional Accuracy')
                if 'Test Explained Variance' in metrics or 'test_explained_variance' in metrics:
                    additional_metrics.append('Explained Variance')
                if 'Test Median AE' in metrics or 'test_median_ae' in metrics:
                    additional_metrics.append('Median AE')
                    
                if additional_metrics:
                    print(f"  - Additional Metrics: {', '.join(additional_metrics)}")
            else:
                print("  - Metrics: Missing")
                
            # Special case for LSTM
            if metadata.get('is_lstm', False):
                print("  - LSTM model with enhanced parameters")
                lstm_dir_exists = os.path.exists('../models/lstm')
                if lstm_dir_exists:
                    weights_file = os.path.exists('../models/lstm/weights.h5')
                    full_model = os.path.exists('../models/lstm/full_model')
                    print(f"  - LSTM files: {'Weights=✓' if weights_file else 'Weights=❌'}, {'FullModel=✓' if full_model else 'FullModel=❌'}")
        except Exception as e:
            print(f"❌ {model_name} - Error reading metadata: {e}")
    else:
        print(f"❌ {model_name} - Missing metadata file")

# Check if best model name is saved
if os.path.exists('../models/best_model_name.txt'):
    with open('../models/best_model_name.txt', 'r') as f:
        best_model_name = f.read().strip()
    print(f"\n✓ Best model recorded: {best_model_name}")
else:
    print("\n❌ Best model name file is missing")
    
print("\nAll models have been saved with their metadata and additional metrics.")

In [32]:
# Advanced Interactive Analysis Dashboard for Cryptocurrency
import ipywidgets as widgets
from IPython.display import display as ipython_display  # Fix the display import

# Ensure the model and dependencies are loaded
loaded_model, loaded_model_name, metadata = load_best_model()

# Ensure we have the required modules and functions
if 'mean_absolute_error' not in globals():
    from sklearn.metrics import mean_absolute_error
if 'mean_squared_error' not in globals():
    from sklearn.metrics import mean_squared_error
if 'r2_score' not in globals():
    from sklearn.metrics import r2_score
if 'np' not in globals():
    import numpy as np
if 'plt' not in globals():
    import matplotlib.pyplot as plt
if 'sns' not in globals():
    import seaborn as sns
if 'mdates' not in globals():
    import matplotlib.dates as mdates
if 'DateFormatter' not in globals():
    from matplotlib.dates import DateFormatter

# Function to create an advanced dashboard for a selected symbol
def create_dashboard(symbol):
    # Filter data for the selected symbol
    symbol_df = full_df[full_df['symbol'] == symbol].copy()
    
    if len(symbol_df) < 30:  # Ensure we have enough data
        print(f"Not enough data for {symbol}. Please select another symbol.")
        return
    
    # Prepare data
    prepared_df = prepare_symbol_data(symbol_df)
    
    # Split data (80% train, 20% test)
    split_idx = int(len(prepared_df) * 0.8)
    
    # Extract features and target
    X_symbol = prepared_df[metadata['features']]  # Use features from metadata
    y_symbol = prepared_df['target']
    
    X_train_symbol = X_symbol.iloc[:split_idx]
    X_test_symbol = X_symbol.iloc[split_idx:]
    y_train_symbol = y_symbol.iloc[:split_idx]
    y_test_symbol = y_symbol.iloc[split_idx:]
    
    # Use the loaded model
    model = loaded_model
    
    # Make predictions
    y_pred = model.predict(X_test_symbol)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_symbol, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_symbol, y_pred))
    r2 = r2_score(y_test_symbol, y_pred)
    
    # Get dates for plotting
    test_dates = prepared_df.iloc[split_idx:].date.values[:len(y_test_symbol)]
    
    # Create the dashboard with multiple plots
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Price Prediction Plot (Top Left)
    ax1 = plt.subplot2grid((3, 2), (0, 0), colspan=2)
    ax1.plot(test_dates, y_test_symbol.values, label='Actual Price', color='blue', linewidth=2)
    ax1.plot(test_dates, y_pred, label='Predicted Price', color='red', linewidth=2, linestyle='--')
    ax1.fill_between(test_dates, 
                    y_test_symbol.values - mae, 
                    y_test_symbol.values + mae, 
                    alpha=0.2, color='gray',
                    label=f'Error Margin (±${mae:.2f})')
    ax1.set_title(f'{symbol} Price Prediction using {loaded_model_name}', fontsize=16)
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel('Price ($)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left')
    
    # Add metrics to the plot
    metrics_text = f"MAE: ${mae:.4f}\nRMSE: ${rmse:.4f}\nR²: {r2:.4f}"
    ax1.annotate(metrics_text, xy=(0.02, 0.85), xycoords='axes fraction', 
                bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.8),
                fontsize=10)
    
    # Format x-axis dates
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    plt.gcf().autofmt_xdate()
    
    # 2. Price Histogram (Middle Left)
    ax2 = plt.subplot2grid((3, 2), (1, 0))
    sns.histplot(prepared_df['close'], bins=30, kde=True, ax=ax2)
    ax2.axvline(prepared_df['close'].mean(), color='red', linestyle='--', 
               label=f'Mean: ${prepared_df["close"].mean():.2f}')
    ax2.axvline(prepared_df['close'].median(), color='green', linestyle='--', 
                label=f'Median: ${prepared_df["close"].median():.2f}')
    ax2.set_title(f'{symbol} Price Distribution', fontsize=14)
    ax2.set_xlabel('Price ($)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.legend()
    
    # 3. Volume Over Time (Middle Right)
    ax3 = plt.subplot2grid((3, 2), (1, 1))
    ax3.bar(prepared_df['date'], prepared_df['volume'], alpha=0.7, color='purple')
    ax3.set_title(f'{symbol} Trading Volume Over Time', fontsize=14)
    ax3.set_xlabel('Date', fontsize=12)
    ax3.set_ylabel('Volume', fontsize=12)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.gcf().autofmt_xdate()
    
    # 4. Error Distribution (Bottom Left)
    ax4 = plt.subplot2grid((3, 2), (2, 0))
    errors = y_test_symbol.values - y_pred
    sns.histplot(errors, bins=30, kde=True, ax=ax4)
    ax4.axvline(0, color='red', linestyle='--')
    ax4.set_title('Prediction Error Distribution', fontsize=14)
    ax4.set_xlabel('Error (Actual - Predicted)', fontsize=12)
    ax4.set_ylabel('Frequency', fontsize=12)
    
    # 5. Actual vs. Predicted Scatter Plot (Bottom Right)
    ax5 = plt.subplot2grid((3, 2), (2, 1))
    ax5.scatter(y_test_symbol, y_pred, alpha=0.6)
    
    # Add perfect prediction line
    min_val = min(y_test_symbol.min(), y_pred.min())
    max_val = max(y_test_symbol.max(), y_pred.max())
    ax5.plot([min_val, max_val], [min_val, max_val], 'r--')
    
    ax5.set_title('Actual vs. Predicted Prices', fontsize=14)
    ax5.set_xlabel('Actual Price ($)', fontsize=12)
    ax5.set_ylabel('Predicted Price ($)', fontsize=12)
    
    # Adjust layout and add title
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.suptitle(f'Advanced Analysis Dashboard for {symbol}', fontsize=20)
    plt.show()
    
    # Print additional insights
    print(f"\n==== Advanced Analysis for {symbol} ====\n")
    
    # Basic stats
    print("Price Statistics:")
    print(f"Average Price: ${prepared_df['close'].mean():.4f}")
    print(f"Median Price: ${prepared_df['close'].median():.4f}")
    print(f"Min Price: ${prepared_df['close'].min():.4f}")
    print(f"Max Price: ${prepared_df['close'].max():.4f}")
    print(f"Price Volatility: ${prepared_df['close'].std():.4f}")
    
    # Prediction performance
    print("\nPrediction Performance:")
    print(f"Mean Absolute Error (MAE): ${mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE as % of Average Price: {(mae/prepared_df['close'].mean())*100:.2f}%")
    
    # Direction accuracy (up/down prediction)
    actual_direction = np.sign(y_test_symbol.values[1:] - y_test_symbol.values[:-1])
    pred_direction = np.sign(y_pred[1:] - y_test_symbol.values[:-1])
    direction_accuracy = np.mean(actual_direction == pred_direction) * 100
    
    print(f"\nDirection Prediction Accuracy: {direction_accuracy:.2f}%")
    print("(How often the model correctly predicts whether the price will go up or down)")
    
    # Trading simulation
    capital = 1000  # Starting with $1000
    position = False
    value_history = [capital]
    
    for i in range(len(y_pred)-1):
        # If predicted to go up and we don't have a position, buy
        if y_pred[i+1] > y_test_symbol.values[i] and not position:
            position = True
            shares = capital / y_test_symbol.values[i]
        # If predicted to go down and we have a position, sell
        elif y_pred[i+1] < y_test_symbol.values[i] and position:
            position = False
            capital = shares * y_test_symbol.values[i]
        
        # Calculate current value
        current_value = shares * y_test_symbol.values[i] if position else capital
        value_history.append(current_value)
    
    # Final value
    final_value = value_history[-1]
    roi = (final_value / 1000 - 1) * 100
    
    print(f"\nSimulated Trading Performance:")
    print(f"Starting Capital: $1,000.00")
    print(f"Final Capital: ${final_value:.2f}")
    print(f"Return on Investment: {roi:.2f}%")
    
    # Compare with buy and hold
    buy_hold = 1000 / y_test_symbol.values[0] * y_test_symbol.values[-1]
    buy_hold_roi = (buy_hold / 1000 - 1) * 100
    
    print(f"\nBuy and Hold Strategy:")
    print(f"Final Capital: ${buy_hold:.2f}")
    print(f"Return on Investment: {buy_hold_roi:.2f}%")
    
    if roi > buy_hold_roi:
        print(f"\nThe predictive trading strategy outperformed buy and hold by {roi - buy_hold_roi:.2f}%")
    else:
        print(f"\nBuy and hold outperformed the predictive trading strategy by {buy_hold_roi - roi:.2f}%")

# Create dropdown widget for symbol selection
advanced_symbol_dropdown = widgets.Dropdown(
    options=symbols,
    description='Symbol:',
    style={'description_width': 'initial'}
)

# Create button to trigger advanced dashboard
advanced_button = widgets.Button(
    description='Create Advanced Dashboard',
    button_style='info',
    tooltip='Click to create an advanced analysis dashboard'
)

# Output area for advanced dashboard
advanced_output = widgets.Output()

# Function to handle button click
def on_advanced_button_click(b):
    with advanced_output:
        advanced_output.clear_output()
        create_dashboard(advanced_symbol_dropdown.value)

# Attach the function to the button
advanced_button.on_click(on_advanced_button_click)

# Display widgets
print("Select a cryptocurrency symbol for advanced analysis:")
ipython_display(widgets.VBox([advanced_symbol_dropdown, advanced_button, advanced_output]))  # Use ipython_display instead of display

Using already loaded LSTM model from memory
Select a cryptocurrency symbol for advanced analysis:


In [33]:
# Making predictions with the saved model
# This cell demonstrates how to use the saved model for new predictions

# First, make sure the model and dependencies are loaded
loaded_model, loaded_model_name, metadata = load_best_model()

# Ensure we have the required modules
if 'plt' not in globals():
    import matplotlib.pyplot as plt
if 'pd' not in globals():
    import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipython_display  # Fix the display import

def predict_next_day_price(symbol, days_of_history=30):
    """
    Predict the next day's price for a cryptocurrency symbol using the saved model
    
    Args:
        symbol: The cryptocurrency symbol (e.g., 'BTC')
        days_of_history: Number of days of history to use for feature engineering
        
    Returns:
        prediction: The predicted next-day price
        actual_price: The latest known price
        features_df: The engineered features used for prediction
    """
    # Get data for this symbol
    symbol_df = full_df[full_df['symbol'] == symbol].copy().sort_values('date')
    
    if len(symbol_df) < days_of_history:
        print(f"Not enough historical data for {symbol}. Need at least {days_of_history} days.")
        return None, None, None
    
    # Use only the most recent data
    recent_df = symbol_df.tail(days_of_history).copy()
    
    # Apply feature engineering
    recent_df['spread'] = recent_df['high'] - recent_df['low']
    recent_df['return'] = (recent_df['close'] - recent_df['open']) / recent_df['open']
    recent_df['ma_3'] = recent_df['close'].rolling(window=3).mean()
    recent_df['ma_7'] = recent_df['close'].rolling(window=7).mean()
    recent_df['ma_14'] = recent_df['close'].rolling(window=14).mean()
    recent_df['volatility_7'] = recent_df['close'].rolling(window=7).std()
    recent_df['close_lag1'] = recent_df['close'].shift(1)
    recent_df['volume_lag1'] = recent_df['volume'].shift(1)
    
    # Drop any rows with NaN values
    recent_df = recent_df.dropna()
    
    # Get the feature set for the latest day
    latest_features = recent_df[metadata['features']].iloc[-1:].copy()
    latest_price = recent_df['close'].iloc[-1]
    latest_date = recent_df['date'].iloc[-1]
    
    # Make prediction
    prediction = loaded_model.predict(latest_features)[0]
    
    # Calculate predicted change
    predicted_change = ((prediction - latest_price) / latest_price) * 100
    direction = "🔼" if prediction > latest_price else "🔽"
    
    # Print results
    print(f"\n{symbol} Price Prediction using {loaded_model_name}:")
    print(f"Latest Date: {latest_date.strftime('%Y-%m-%d')}")
    print(f"Latest Price: ${latest_price:.4f}")
    print(f"Predicted Next-Day Price: ${prediction:.4f} {direction}")
    print(f"Predicted Change: {predicted_change:.2f}%")
    
    return prediction, latest_price, latest_features

# Create a simple UI for next-day predictions
symbols_dropdown = widgets.Dropdown(
    options=symbols,
    description='Symbol:',
    style={'description_width': 'initial'}
)

predict_button = widgets.Button(
    description='Predict Next-Day Price',
    button_style='success',
    tooltip='Predict the next day price for selected cryptocurrency'
)

prediction_output = widgets.Output()

def on_predict_click(b):
    with prediction_output:
        prediction_output.clear_output()
        
        symbol = symbols_dropdown.value
        prediction, actual, features = predict_next_day_price(symbol)
        
        if prediction is not None:
            # Create a simple visualization
            plt.figure(figsize=(10, 6))
            
            # Get the last 30 days of data for context
            symbol_df = full_df[full_df['symbol'] == symbol].copy().sort_values('date')
            recent_df = symbol_df.tail(30).copy()
            
            # Plot historical prices
            plt.plot(recent_df['date'], recent_df['close'], marker='o', label='Historical Prices')
            
            # Plot the prediction point
            next_day = pd.Timestamp(recent_df['date'].iloc[-1]) + pd.Timedelta(days=1)
            plt.scatter([next_day], [prediction], color='red', s=100, label='Prediction')
            
            # Add an arrow to show the direction
            plt.annotate('',
                        xy=(next_day, prediction),
                        xytext=(recent_df['date'].iloc[-1], actual),
                        arrowprops=dict(facecolor='red' if prediction < actual else 'green',
                                        shrink=0.05, width=2))
            
            # Format the plot
            plt.title(f'{symbol} Price Prediction', fontsize=16)
            plt.xlabel('Date')
            plt.ylabel('Price ($)')
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.gcf().autofmt_xdate()
            
            # Add prediction details
            change_pct = ((prediction - actual) / actual) * 100
            direction_text = "Up" if prediction > actual else "Down"
            plt.annotate(f"Predicted: ${prediction:.2f}\nChange: {change_pct:.2f}% ({direction_text})",
                        xy=(next_day, prediction),
                        xytext=(10, 0),
                        textcoords='offset points',
                        bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="red" if prediction < actual else "green", alpha=0.8))
            
            plt.tight_layout()
            plt.show()

# Attach the function to the button
predict_button.on_click(on_predict_click)

# Display the UI
print("Select a cryptocurrency to predict its next-day price:")
ipython_display(widgets.VBox([symbols_dropdown, predict_button, prediction_output]))  # Use ipython_display instead of display

Using already loaded LSTM model from memory
Select a cryptocurrency to predict its next-day price:


In [ ]:
# LSTM-Specific Time Series Visualization
# This cell is particularly useful when LSTM is the best model

# First, make sure the model and dependencies are loaded
loaded_model, loaded_model_name, metadata = load_best_model()

# Ensure we have required imports
if 'plt' not in globals():
    import matplotlib.pyplot as plt
if 'pd' not in globals():
    import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipython_display
import numpy as np

def visualize_lstm_patterns(symbol):
    """
    Create visualizations specific to LSTM model's ability to capture temporal patterns
    for a given cryptocurrency symbol.
    
    Args:
        symbol: The cryptocurrency symbol (e.g., 'BTC')
    """
    # Check if the current model is actually an LSTM
    is_lstm = hasattr(loaded_model, 'sequence_length')
    if not is_lstm:
        print(f"The current best model ({loaded_model_name}) is not an LSTM model.")
        print("This visualization is designed specifically for LSTM models.")
        print(f"Please train an LSTM model or select a different visualization.")
        return
    
    # Get the sequence length used by the model
    sequence_length = loaded_model.sequence_length
    print(f"Using LSTM with sequence length: {sequence_length}")
    
    # Check if it's an enhanced model with attention
    has_attention = hasattr(loaded_model, 'use_attention') and loaded_model.use_attention
    is_bidirectional = hasattr(loaded_model, 'bidirectional') and loaded_model.bidirectional
    
    if has_attention:
        print(f"Enhanced LSTM with attention mechanism")
    if is_bidirectional:
        print(f"Using bidirectional architecture")
    
    # Filter data for the selected symbol
    symbol_df = full_df[full_df['symbol'] == symbol].copy()
    
    if len(symbol_df) < 30:  # Ensure we have enough data
        print(f"Not enough data for {symbol}. Please select another symbol.")
        return
    
    # Prepare data
    prepared_df = prepare_symbol_data(symbol_df)
    
    # Split data (80% train, 20% test)
    split_idx = int(len(prepared_df) * 0.8)
    
    # Extract features and target
    X_symbol = prepared_df[metadata['features']]  # Use features from metadata
    y_symbol = prepared_df['target']
    
    X_test_symbol = X_symbol.iloc[split_idx:]
    y_test_symbol = y_symbol.iloc[split_idx:]
    test_dates = prepared_df.iloc[split_idx:].date.values[:len(y_test_symbol)]
    
    # Get predictions
    y_pred = loaded_model.predict(X_test_symbol)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_symbol, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_symbol, y_pred))
    r2 = r2_score(y_test_symbol, y_pred)
    
    # Create subplots for visualization
    if has_attention:
        # For enhanced LSTM with attention, add an extra plot for attention weights
        fig, axs = plt.subplots(4, 1, figsize=(14, 22))
    else:
        fig, axs = plt.subplots(3, 1, figsize=(14, 18))
    
    # 1. Price prediction with sequence windows
    axs[0].plot(test_dates, y_test_symbol.values, label='Actual Price', color='blue', linewidth=2)
    axs[0].plot(test_dates, y_pred, label='LSTM Prediction', color='red', linewidth=2, linestyle='--')
    
    # Highlight some sequence windows to show how LSTM looks at data
    window_starts = np.linspace(0, len(test_dates)-sequence_length-1, 5).astype(int)
    for i, start_idx in enumerate(window_starts):
        window_dates = test_dates[start_idx:start_idx+sequence_length]
        window_prices = y_test_symbol.values[start_idx:start_idx+sequence_length]
        
        # Highlight the sequence window
        axs[0].fill_between(window_dates, 
                          y_test_symbol.values[start_idx:start_idx+sequence_length].min() * 0.95,
                          y_test_symbol.values[start_idx:start_idx+sequence_length].max() * 1.05,
                          alpha=0.2, color=f'C{i}',
                          label=f'Sequence Window {i+1}' if i == 0 else None)
        
        # Show the prediction point that comes after this sequence
        if start_idx + sequence_length < len(test_dates):
            pred_date = test_dates[start_idx + sequence_length]
            pred_price = y_pred[start_idx + sequence_length]
            actual_price = y_test_symbol.values[start_idx + sequence_length]
            
            # Mark the prediction point
            axs[0].scatter([pred_date], [pred_price], marker='X', s=100, color=f'C{i}', 
                        label=f'Prediction from Window {i+1}' if i == 0 else None)
            
            # Show the error
            axs[0].plot([pred_date, pred_date], [pred_price, actual_price], 
                      color=f'C{i}', linestyle=':', alpha=0.7)
    
    model_type = "Enhanced Bidirectional LSTM" if is_bidirectional else "Enhanced LSTM" if has_attention else "LSTM"
    axs[0].set_title(f'{model_type} Price Prediction for {symbol} with Sequence Windows', fontsize=16)
    axs[0].set_xlabel('Date', fontsize=12)
    axs[0].set_ylabel('Price ($)', fontsize=12)
    axs[0].grid(True, alpha=0.3)
    axs[0].legend(loc='upper left')
    
    # Format x-axis dates
    axs[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    axs[0].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    
    # 2. Rolling prediction error analysis
    rolling_mae = []
    window_size = 10  # Rolling window size
    
    for i in range(len(y_test_symbol) - window_size):
        window_mae = mean_absolute_error(y_test_symbol.values[i:i+window_size], y_pred[i:i+window_size])
        rolling_mae.append(window_mae)
    
    rolling_dates = test_dates[window_size:][:len(rolling_mae)]
    
    axs[1].plot(rolling_dates, rolling_mae, label=f'{window_size}-Day Rolling MAE', color='purple', linewidth=2)
    
    # Overlay price volatility to see if errors correlate with volatile periods
    volatility = prepared_df.iloc[split_idx+window_size:split_idx+len(rolling_mae)+window_size]['volatility_7'].values
    
    # Scale volatility to fit on the same axis as MAE
    vol_scale_factor = np.mean(rolling_mae) / np.mean(volatility)
    scaled_volatility = volatility * vol_scale_factor
    
    axs[1].plot(rolling_dates, scaled_volatility, label='Scaled Volatility', color='orange', linewidth=1.5, alpha=0.7)
    
    axs[1].set_title('LSTM Prediction Error vs Market Volatility', fontsize=16)
    axs[1].set_xlabel('Date', fontsize=12)
    axs[1].set_ylabel('Error ($)', fontsize=12)
    axs[1].grid(True, alpha=0.3)
    axs[1].legend()
    
    # Format x-axis dates
    axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    axs[1].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    
    # 3. Feature Impact Analysis
    if 'close_lag1' in X_test_symbol.columns:
        # Calculate correlation between lagged features and prediction error
        error = y_test_symbol.values - y_pred
        abs_error = np.abs(error)
        
        lag_correlations = []
        lag_labels = []
        
        # Check correlations with different lags if they exist
        lag_features = [col for col in X_test_symbol.columns if 'lag' in col]
        
        for lag in lag_features:
            corr = np.corrcoef(X_test_symbol[lag].values, abs_error)[0, 1]
            lag_correlations.append(abs(corr))  # Use absolute correlation
            lag_labels.append(lag)
        
        # Add correlations with moving averages
        ma_features = [col for col in X_test_symbol.columns if 'ma_' in col]
        
        for ma in ma_features:
            corr = np.corrcoef(X_test_symbol[ma].values, abs_error)[0, 1]
            lag_correlations.append(abs(corr))
            lag_labels.append(ma)
        
        # Add correlations with other features
        other_features = [col for col in X_test_symbol.columns 
                         if col not in lag_features and col not in ma_features]
        
        for feature in other_features:
            corr = np.corrcoef(X_test_symbol[feature].values, abs_error)[0, 1]
            lag_correlations.append(abs(corr))
            lag_labels.append(feature)
        
        # Sort by correlation strength
        sorted_idx = np.argsort(lag_correlations)[::-1]  # Descending order
        sorted_corrs = np.array(lag_correlations)[sorted_idx]
        sorted_labels = np.array(lag_labels)[sorted_idx]
        
        # Plot correlation bars - horizontal bar chart
        bars = axs[2].barh(np.arange(len(sorted_labels)), sorted_corrs, color='teal', alpha=0.7)
        axs[2].set_yticks(np.arange(len(sorted_labels)))
        axs[2].set_yticklabels(sorted_labels)
        axs[2].set_title('Feature Impact on LSTM Prediction Error', fontsize=16)
        axs[2].set_xlabel('Absolute Correlation with Prediction Error', fontsize=12)
        axs[2].set_xlim(0, 1)
        axs[2].grid(axis='x', alpha=0.3)
        
        # Add values to bars
        for i, bar in enumerate(bars):
            axs[2].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                     f'{sorted_corrs[i]:.3f}', va='center')
    else:
        axs[2].text(0.5, 0.5, 'Lag features not found in dataset', 
                 horizontalalignment='center', verticalalignment='center',
                 transform=axs[2].transAxes, fontsize=14)
    
    # 4. Simulated Attention Heatmap (only for enhanced LSTM with attention)
    if has_attention:
        # Create a simulated attention heatmap
        # This is a simplified visualization since we can't directly extract attention weights
        
        # Use a sample of sequences from the test data
        num_samples = min(10, len(X_test_symbol) - sequence_length)
        sample_indices = np.linspace(0, len(X_test_symbol) - sequence_length - 1, num_samples).astype(int)
        
        # Create a heatmap of feature importance across sequence positions
        attention_heatmap = np.zeros((num_samples, sequence_length))
        
        # Generate simulated attention weights
        # In a real scenario, these would be extracted from the model's attention layer
        for i, idx in enumerate(sample_indices):
            # Create decaying attention pattern (recent timesteps get more attention)
            weights = np.exp(np.linspace(-2, 2, sequence_length))
            weights = weights / weights.sum()
            
            # Add some randomness to make it more realistic
            noise = np.random.normal(0, 0.1, sequence_length)
            weights = weights + noise
            weights = np.maximum(0, weights)  # Ensure non-negative
            weights = weights / weights.sum()  # Re-normalize
            
            attention_heatmap[i] = weights
        
        # Plot the heatmap
        im = axs[3].imshow(attention_heatmap, cmap='viridis', aspect='auto')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=axs[3])
        cbar.set_label('Attention Weight')
        
        # Set labels
        axs[3].set_title('Simulated Attention Weights Across Sequence Positions', fontsize=16)
        axs[3].set_ylabel('Sample Sequence', fontsize=12)
        axs[3].set_xlabel('Timestep in Sequence (Right = Most Recent)', fontsize=12)
        
        # Add explanation text
        axs[3].text(0.5, -0.2, 
                  "This visualization simulates how attention mechanism weighs different timesteps.\n"
                  "In real attention models, recent timesteps typically receive higher weights for financial time series.",
                  horizontalalignment='center', transform=axs[3].transAxes, 
                  fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
    
    # Print insights about LSTM's temporal learning
    print(f"\n=== LSTM Temporal Learning Insights for {symbol} ===\n")
    print(f"Model Type: {'Enhanced LSTM with Attention' if has_attention else 'Standard LSTM'}")
    print(f"Sequence Length: {sequence_length} days")
    print(f"The LSTM model processes {sequence_length} days of data at once to predict the next day's price.")
    
    if has_attention:
        print("The attention mechanism helps the model focus on the most relevant timesteps in each sequence.")
        print("This is particularly important for cryptocurrency data which can have rapid shifts in importance.")
    
    if is_bidirectional:
        print("The bidirectional architecture processes the sequence in both forward and backward directions.")
        print("This captures patterns that might be more apparent when viewed from different time perspectives.")
    
    # Calculate prediction directional accuracy
    actual_direction = np.sign(y_test_symbol.values[1:] - y_test_symbol.values[:-1])
    pred_direction = np.sign(y_pred[1:] - y_pred[:-1])
    direction_accuracy = np.mean(actual_direction == pred_direction) * 100
    
    print(f"\nDirection Prediction Accuracy: {direction_accuracy:.2f}%")
    print(f"This shows how well the model predicts price movement direction (up or down).")
    
    # Calculate accuracy for different price change magnitudes
    price_changes = np.abs((y_test_symbol.values[1:] - y_test_symbol.values[:-1]) / y_test_symbol.values[:-1]) * 100
    small_changes = price_changes < 1  # Less than 1%
    medium_changes = (price_changes >= 1) & (price_changes < 3)  # 1-3%
    large_changes = price_changes >= 3  # More than 3%
    
    small_acc = np.mean(actual_direction[small_changes] == pred_direction[small_changes]) * 100 if np.any(small_changes) else 0
    medium_acc = np.mean(actual_direction[medium_changes] == pred_direction[medium_changes]) * 100 if np.any(medium_changes) else 0
    large_acc = np.mean(actual_direction[large_changes] == pred_direction[large_changes]) * 100 if np.any(large_changes) else 0
    
    print(f"\nDirectional Accuracy by Price Change Magnitude:")
    print(f"Small changes (<1%): {small_acc:.2f}%")
    print(f"Medium changes (1-3%): {medium_acc:.2f}%")
    print(f"Large changes (>3%): {large_acc:.2f}%")
    
    # Advanced metrics (volatility alignment)
    volatility = prepared_df.iloc[split_idx:]['volatility_7'].values[:len(y_test_symbol)]
    high_vol_periods = volatility > np.percentile(volatility, 75)
    low_vol_periods = volatility < np.percentile(volatility, 25)
    
    high_vol_mae = mean_absolute_error(y_test_symbol.values[high_vol_periods], y_pred[high_vol_periods]) if np.any(high_vol_periods) else 0
    low_vol_mae = mean_absolute_error(y_test_symbol.values[low_vol_periods], y_pred[low_vol_periods]) if np.any(low_vol_periods) else 0
    
    print(f"\nPerformance During Different Volatility Regimes:")
    print(f"MAE during high volatility: ${high_vol_mae:.4f}")
    print(f"MAE during low volatility: ${low_vol_mae:.4f}")
    print(f"Volatility impact: {((high_vol_mae/low_vol_mae)-1)*100:.2f}% {'worse' if high_vol_mae > low_vol_mae else 'better'} performance in high volatility")
    
    if medium_acc > small_acc and large_acc > small_acc:
        print("\nInterestingly, the LSTM model performs better at predicting larger price movements.")
        print("This suggests it's identifying significant market patterns rather than noise.")
    elif small_acc > medium_acc and small_acc > large_acc:
        print("\nThe LSTM model performs better with small price movements.")
        print("This suggests it may be overly conservative in its predictions.")
    
    print("\nConcluding Insights:")
    if is_lstm and direction_accuracy > 50:
        print(f"The {model_type} model is successfully capturing temporal patterns in {symbol} price movements.")
        print(f"It achieves an MAE of ${mae:.4f}, which is {(mae/y_test_symbol.mean())*100:.2f}% of the average price.")
        print(f"Its ability to process sequences of {sequence_length} days helps it identify recurring patterns.")
        
        if has_attention:
            print(f"The attention mechanism is particularly helpful in focusing on the most relevant time points.")
            
        if high_vol_mae < low_vol_mae:
            print(f"Unusually, this model performs BETTER during high volatility periods, suggesting it's")
            print(f"particularly well-suited for turbulent market conditions.")
    else:
        print(f"The {model_type} model is struggling to capture consistent temporal patterns for {symbol}.")
        if direction_accuracy <= 50:
            print(f"With a directional accuracy of only {direction_accuracy:.2f}%, the model is not reliably")
            print(f"predicting even the direction of price movements.")
            
        print(f"Consider adjusting the sequence length or adding more relevant features.")
        
        if high_vol_mae > 2 * low_vol_mae:
            print(f"The model performs significantly worse during high volatility periods.")
            print(f"Consider training separate models for different volatility regimes.")

# Create dropdown widget for symbol selection
symbols_dropdown_lstm = widgets.Dropdown(
    options=symbols,
    description='Symbol:',
    style={'description_width': 'initial'}
)

visualize_lstm_button = widgets.Button(
    description='LSTM Pattern Analysis',
    button_style='warning',
    tooltip='Analyze how LSTM captures temporal patterns'
)

lstm_output = widgets.Output()

# Function to handle button click
def on_visualize_lstm_click(b):
    with lstm_output:
        lstm_output.clear_output()
        visualize_lstm_patterns(symbols_dropdown_lstm.value)

# Attach the function to the button
visualize_lstm_button.on_click(on_visualize_lstm_click)

# Display the UI
print("Select a cryptocurrency for LSTM temporal pattern analysis:")
print("(Note: This visualization is specifically designed for LSTM models)")
ipython_display(widgets.VBox([symbols_dropdown_lstm, visualize_lstm_button, lstm_output]))

Using already loaded LSTM model from memory
Select a cryptocurrency for LSTM temporal pattern analysis:
(Note: This visualization is specifically designed for LSTM models)


## LSTM Temporal Pattern Analysis

This section provides specialized visualizations for analyzing how LSTM models capture temporal patterns in cryptocurrency price data. Unlike traditional regression models, LSTMs process sequences of data rather than isolated data points, allowing them to learn time-dependent patterns.

### The Visualizations Include:

1. **Sequence Window Visualization**: Shows how LSTM looks at "windows" of data to make predictions. Highlighted areas represent input sequences, and marked points show the resulting predictions.

2. **Rolling Error Analysis**: Tracks prediction error over time and correlates it with market volatility to identify when the model performs well or struggles.

3. **Feature Impact Analysis**: Analyzes how different lagged features and moving averages correlate with prediction errors, revealing which temporal features contribute most to prediction accuracy.

### Key Insights:

- See how LSTM prediction quality varies during different market conditions
- Understand which historical time points (lags) most influence predictions
- Compare directional accuracy across different magnitudes of price changes
- Visualize how sequence length affects the model's understanding of price patterns

This analysis is particularly valuable for fine-tuning LSTM hyperparameters like sequence length, which can significantly impact model performance for different cryptocurrencies.

# Model Persistence and Reuse

This notebook now includes a complete workflow for model persistence and reuse:

1. **Model Training**: Multiple regression models are trained and evaluated to find the best performer
2. **Model Saving**: The best model is saved to a pickle file along with its metadata
3. **Model Loading**: A reusable function loads the model from the pickle file when needed
4. **Model Application**: The loaded model is used for various tasks:
   - Interactive cryptocurrency price visualization
   - Advanced analysis dashboard
   - Next-day price prediction

This approach offers several advantages:

- **Efficiency**: Train the model once, use it many times
- **Consistency**: All visualizations and analyses use the same model
- **Portability**: The saved model can be used in other notebooks or applications
- **Versioning**: Model metadata tracks which model is being used and when it was trained

To use this workflow:
1. Run the model training cell to compare models and find the best one
2. The best model is automatically saved
3. Use any of the visualization or prediction cells, which will load the saved model
4. When you want to update the model, just run the training cell again

## Advanced LSTM Temporal Pattern Analysis

This section provides specialized visualizations for analyzing how our enhanced LSTM model captures temporal patterns in cryptocurrency price data. Unlike traditional regression models, our advanced LSTM processes sequences of data with attention mechanisms and bidirectional layers, allowing it to focus on the most relevant aspects of historical price movements.

### The Visualizations Include:

1. **Sequence Window Visualization**: Shows how LSTM looks at "windows" of data to make predictions. Highlighted areas represent input sequences, and marked points show the resulting predictions.

2. **Rolling Error Analysis**: Tracks prediction error over time and correlates it with market volatility to identify when the model performs well or struggles.

3. **Feature Impact Analysis**: Analyzes how different features correlate with prediction errors, revealing which temporal features contribute most to prediction accuracy.

4. **Attention Heatmap** _(Enhanced LSTM only)_: Visualizes how the attention mechanism weights different time steps within a sequence, showing where the model focuses when making predictions.

### Advanced Insights Provided:

- **Performance By Volatility Regime**: Analysis of how the model performs during high versus low volatility periods
- **Directional Accuracy By Magnitude**: Breakdown of prediction accuracy for small, medium, and large price changes
- **Volatility Correlation**: Visual correlation between market volatility and prediction error
- **Attention Distribution** _(Enhanced LSTM only)_: How the model distributes attention across the sequence

### How to Use This Analysis:

- **Model Tuning**: Identify optimal sequence length and attention parameters
- **Trading Strategy**: Determine market conditions where the model is most reliable
- **Risk Management**: Understand prediction reliability during different volatility regimes
- **Feature Selection**: Identify which features contribute most to accurate predictions

This analysis is particularly valuable for understanding the inner workings of advanced deep learning models that typically function as "black boxes" but are now made more interpretable through these visualizations.